## DSPy Quantized Qwen2 Information Extraction

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from rich import print
import pandas as pd
import ast
import torch
import gc


from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch

from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2','rougeL'], use_stemmer=True)
import json

#### Helper Functions

In [2]:
def validate_ans(example, pred, trace = None):

    gold = re.sub(r'\n|\s+ ', '',dict(example)['answer']).lower()
    print(gold)
    torch.cuda.empty_cache()
    gc.collect()
    prediction = str(json.loads(pred.answer.split('Answer: ')[1].lower()))
    torch.cuda.empty_cache()
    from torch import bfloat16
    gc.collect()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Test examples

In [3]:
test_examples = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\Manual Labeling - Sheet2.csv", header = None)

#### Set LLM Qwen2

In [10]:
model_name = "Qwen/Qwen2-7B"
llm = dspy.HFModel(model=model_name, hf_device_map='auto', model_kwargs= {'temperature': 0.0, 'do_sample': False})
llm.model=None
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=torch.bfloat16  # Computation type
)
llm.model=AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
dspy.settings.configure(lm=llm)

#### Create DSPy Signature and Module

In [10]:
class GenerateAnswer(dspy.Signature):
    """Extract information from a job posting and return the output in a json format if you don't know answer Not Specified. Should be key-value with output as dictionary."""

    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(desc="key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words")

class QUESTIONANSWER(dspy.Module):
    def __init__(self,question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question=question

    def forward(self, context):
        context = context.replace('\n', ' ').replace('“', '"').replace('”', '"')
        context = normalize(context)
        question=self.question
        pred = self.generate_answer(context=context, question=question)
        torch.cuda.empty_cache()
        gc.collect()
        pred = re.sub(r"```\n|```", "",pred.answer)
        return dspy.Prediction(context=context,answer=pred)

In [11]:
uncompiled_fs=QUESTIONANSWER('''
                            "position_title" : What is the title of this position?
                            "location" : Where is this position located, including city, state and zip code?
                            "work_arrange" : What is the work arrangement for this position, remote, hybrid, or on-site?
                            "experience" : what are years of experience required for this position?
                            "employment_type" : What is the employment type, full time, part time, or internship?
                            "pay" : What is the pay for this position?
                            "degree" : What is required degree?
                            "certifications" : What certifications or qualifications are required?
                            "required_skills" : What are required skills?
                              ''')

#### Test Uncompiled DSPy 

In [12]:
print(test_examples[1][2])

NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility 
Inc. compensation: HOURLY - EVERY OTHER WEEKEND REQUIRED employment type: full-time job title: NURSE URSES - RNs or
LPNs Sudbury Pines Extended Care Sudbury, MA 01776 Sudbury Pines Extended Care Facility is seeking Nurses to join 
our team! Currently seeking the following positions: Full time or Part Time RN or LPN Sudbury Pines Extended Care 
facility is a 92 bed facility located in the MetroWest area. We are a single family owned facility who strives to 
provide quality care in a home-like and family oriented environment for our residents. Responsibilities: 
Responsible for the overall nursing care and delivery of resident services - medication pass, treatments, resident 
quality of life, etc. Manage staff and promote staff morale to ensure residents needs are being met in a proactive 
manner. Qualifications: Must have a valid MA Nursing License. Minimum of 1 year Long term care experience/SNF 
experience preferred - although new graduates welcome. We offer great benefits for all Full Time staff -some 
limitations apply for part time staff. Child Day Care facility on site since 1986 - infants through preschoolers - 
prorated for staff. Job Type: Full-time or part-time Job Types: Full-time, Part-time Benefits: 401(k) 401(k) 
matching Dental insurance Flexible schedule Health insurance Life insurance Paid time off Referral program Tuition 
reimbursement Physical setting: JCAHO accredited facility Long term care Nursing home Standard shift: Day shift 
Evening shift Night shift Supplemental schedule: Holidays Overtime Weekly schedule: Rotating weekends COVID-19 
considerations: All staff are required to follow current COVID 19 protocols as defined by the Commonwealth of MA - 
must be prepared to wear masks, and follow other infection control protocols as well as all vaccination guidelines 
expected to be employed in a LTC SNF Ability to commute/relocate: Sudbury, MA 01776: Reliably commute or planning 
to relocate before starting work (Required) Experience: Nursing to: 1 year (Preferred) License/Certification: RN or
LPN License in Massachusetts (Required) Work Location: One location Principals only. Recruiters, please don't 
contact this job poster. Do NOT contact this job poster with unsolicited services or offers. post id: 7694393986 
updated: [ ]

In [13]:
with dspy.context(lm = llm):
    pred = uncompiled_fs(context = test_examples[1][2])
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:693: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{
    'position_title': 'Nurses - RNS or LPNs',
    'location': 'Sudbury, MA 01776',
    'work_arrange': 'On-site',
    'experience': 'Minimum of 1 year long term care experience/SNF experience preferred',
    'employment_type': 'Full-time or Part-time',
    'pay': 'Hourly',
    'degree': 'Valid MA Nursing License',
    'certifications': 'RN or LPN license in Massachusetts',
    'required_skills': 'Nursing'
}

#### Create Test Format

In [14]:
train_example_list = [
    """
    {
        "position_title": "Derrickhand",
        "location": "Buckhannon, West Virginia 26201",
        "work_arrangement": "On-Site, Shifts",
        "experience": "1-2 years of Derrickhand experience",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "High school diploma/GED or equivalent",
        "certification": "CDL B License",
        "required_skills": "Effective verbal/written communication in English, ability to interact with teams in a fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent customer-service"
    }
    """,
    """
    {
        "position_title": "Pharmacy Technician",
        "location": "San Quentin, California",
        "work_arrangement": "On-Site, Shifts, Relocation required if applicable",
        "experience": "1 year of experience as Pharmacy Technician",
        "employment_type": "Contract",
        "pay": "$18-$19/hr",
        "degree": "High school diploma or GED",
        "certification": "Pharmacy Technician Certification, BLS Certification",
        "required_skills": "Excellent communication skills, ability to use computer for day-to-day tasks, basic math for counting medications"
    }
    """,
    """
    {
        "position_title": "GIS Technician",
        "location": "Oklahoma City, OK 73134",
        "work_arrangement": "On-Site, Shifts, Relocation required if applicable",
        "experience": "3-5 years of GIS experience",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "Bachelor's degree in a related field",
        "certification": "Not specified",
        "required_skills": "GIS, ArcPy, ESRI ArcGIS Desktop or ArcPro, Field Maps/ArcGIS Online, Microsoft Office suites, clerical skills, ability to work in a team environment, initiative in recognizing need for improvements of existing systems, tracking down msising/misfiled items, filing accuracy"
    }
    """,
    """
    {
        "position_title": "Graphic Designer",
        "location": "Goochland, VA",
        "work_arrangement": "Hybrid, with two in-office days per week",
        "experience": "Minimum 5 years design and publications experience",
        "employment_type": "Part-time",
        "pay": "Not specified",
        "degree": "College degree in graphic design, visual arts, or related field",
        "certification": "Not specified",
        "required_skills": "proficiency with InDesign, Photoshop, Illustrator, working knowledge of Constant Contact, strong organizational skills, excellent oral/written communication and client-relations skills, ability to work under pressure, working knowledge of AP style, 35mm and digital photography skills, Mac environment"
    }
    """,
    """
    {
        "position_title": "Senior Cybersecurity Analyst",
        "location": "Washington DC, USA",
        "work_arrangement": "Not Specified",
        "experience": "5 years of experience in cybersecurity",
        "employment_type": "Full-time",
        "pay": "Not specified",
        "degree": "Bachelor's degree",
        "certification": "DOD 8570 Level II, DOD 8570 Level III or Manager",
        "required_skills": "carbon black implementation, splunk, CDM dashboards, CI/CD, black box testing of IT assets"
    }
    """,
    """
    {
        "position_title": "Academic Instructor",
        "location": "Fullerton, CA",
        "work_arrangement": "Not specified",
        "experience": "Experience working with low-income and diverse student population, experience working with AUHSD student, experience working with middle or high school students",
        "employment_type": "Part-time",
        "pay": "$47-$52/hr",
        "degree": "Bachelor's degree, Master's degree",
        "certification": "Not specified",
        "required_skills": "Teaching, ability to work in a collaborative team environment, develop effective teaching strategies, lifting of up to 25lbs"
    }
    """,
    """
    {
        "position_title": "Retail Scan Associate",
        "location": "Luverne, Minnesota",
        "work_arrangement": "On-site",
        "experience": "Not specified",
        "employment_type": "Part-time",
        "pay": "$16/hr",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Ability to endure being on your feet for long periods of time, ability to lift up to 25lbs, reach 6 feet in the air, ability to perform repetitive movements with hands, wrists, arms, and legs, attention to detail and ability to work independently"
    }
    """,
    """
    {
        "position_title": "Machinist Operator",
        "location": "Valencia, CA",
        "work_arrangement": "On-site, Shifts",
        "experience": "3 years of CNC operating and programming experience, experience with BobCad",
        "employment_type": "Full-time, Temp-to-Hire",
        "pay": "$30/hr",
        "degree": "High school diploma/GED",
        "certification": "BobCad, Solid Works, HAAS ST 20, Fadal WMC4020, Akira-Seiki SL20",
        "required_skills": "Strong math, problem solving and analytical skills"
    }
    """,
    """
    {
        "position_title": "Automotive Technician",
        "location": "Bremerton, WA",
        "work_arrangement": "On-site",
        "experience": "4-7 years of experience",
        "employment_type": "Full-time",
        "pay": "$50,000 - $83,200 a year",
        "degree": "High school diploma",
        "certification": "Not specified",
        "required_skills": "capable of diagnosing and repairing any system of the automobile to dealership and manufacturer's standards without supervision"
    }
    """,
    """
    {
        "position_title": "Industrial Maintenance Electrician",
        "location": "Cary, NC",
        "work_arrangement": "On-site, Shifts",
        "experience": "previous experience working in a food manufacturing plant",
        "employment_type": "Full-time",
        "pay": "$30.53/hr",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Basic computer skills including Microsoft Office, Demonstrated knowledge of behavior-based safety systems, ability to lift up to 50lbs"
    }
    """,
    """
    {
        "position_title": "Technician",
        "location": "Palm Harbor, FL",
        "work_arrangement": "On-site",
        "experience": "5+ years of service technician experience, 10+ preferred",
        "employment_type": "Full-time",
        "pay": "$27K - $66K",
        "degree": "High school diploma/GED",
        "certification": "ASE Certification, Diagnostic, Electric and Engine Repair",
        "required_skills": "excellent hand-eye coordination, mechanical and troubleshooting skills, ability to operate electronic diagnostic equipment, excellent customer service skills, basic computer competencies, ability to collaborate with others, ability to learn new technology"
    }
    """,
    """
    {
        "position_title": "Director of Email Marketing",
        "location": "Austin, TX",
        "work_arrangement": "Not specified",
        "experience": "5+ years of experience as project manager",
        "employment_type": "Full-time",
        "pay": "$125,000 - $250,000 a year",
        "degree": "Bachelor's degree in marketing",
        "certification": "Not specified",
        "required_skills": "Ability to collaborate with a team of people to learn and grow and own new responsibilities, sophisticated verbal and written communication, people, and leadership skills, advanced analytical and problem-solving skills, focus on email marketing, experience with testing and activating cold audiences, experience with project methodologies including agile, waterfall, and scrum, experience having worked at a startup or a small company with less than 50 people"
    }
    """,
    """
    {
        "position_title": "Campus Store Leader",
        "location": "Philadelphia, PA",
        "work_arrangement": "On-site",
        "experience": "0-5 years of relevant experience, retail experience is a plus",
        "employment_type": "Full-time",
        "pay": "$15.00 - $21.56 an hour",
        "degree": "Associate's degree",
        "certification": "Not specified",
        "required_skills": "analysis skills, computer skills, financial acumen, communication skills, time management, advanced relationship building, ability to influence a team, customer outreach"
    }
    """,
    """
    {
        "position_title": "Dental Laboratory Technician",
        "location": "Wood Dale, IL",
        "work_arrangement": "On-site",
        "experience": "5 years of Fabrication experience",
        "employment_type": "Part-time, Contract",
        "pay": "$14.00 - $22.00 per hour",
        "degree": "High school diploma or equivalent",
        "certification": "Not specified",
        "required_skills": "3Shape software savvyy, Fabrication of custom trays, bite rims, denture/partial repairs"
    }
    """,
   

 """
    {
        "position_title": "Senior Data Engineer (Machine Learning)",
        "location": "Wood Dale, IL",
        "work_arrangement": "Remote",
        "experience": "5+ years of experience as a Data Engineer",
        "employment_type": "Full-time, Direct Hire",
        "pay": "$130,000 - $170,000 per year",
        "degree": "Bachelor's degree in computer science, engineering, or data science",
        "certification": "Not specified",
        "required_skills": "focused on Python, Postgres, and DBT, Experience using Google Cloud Platform and ideally Google AI tools, Experience with data modeling, data warehousing, and building ETL pipelines with DBT, Python, Postgres, DBT, SQL, PostgreML, TensorFlow, PyTorch, Google BigQuery, Airflow, Fivetran, Make, excellent understanding of machine learning algorithms, processes, tools and platforms, proven ability to drive business results with data-based insights"
    }
    """,
    """
    {
        "position_title": "Solutions Architect",
        "location": "Mayfield Heights, OH",
        "work_arrangement": "Not Specified",
        "experience": "Minimum 2 years of experience",
        "employment_type": "Full-time",
        "pay": "$64.7K - $81.9K a year",
        "degree": "Bachelors Degree in Information Technology, MIS, or Financial Accounting, Advanced diploma/degree in Finance/Management Accounting preferred",
        "certification": "CPA Certification preferred, Certification in SAP FICO highly preferred",
        "required_skills": "supporting SAP FI/CO modules and related integration with MM and SD SAP AP, AR, COPA and COPC, Minimum of 2 full SAP life-cycle implementations, and upgrades, Experience with SAP S/4 HANA Upgrade a plus, SAP FI/CO modules, MM, SD, SAP, AP, AR, COPA, COPC"
    }
    """,
    """
    {
        "position_title": "Marketing Manager",
        "location": "Andover, NJ 07821",
        "work_arrangement": "Remote",
        "experience": "Minimum of 2 years in marketing & social media management preferred, demonstrable experience with social analytics tools",
        "employment_type": "Full-time, Shift and schedule weekends as needed, nights as needed",
        "pay": "$60,000 a year",
        "degree": "Bachelor's degree in marketing, communications, or related field",
        "certification": "Not specified",
        "required_skills": "excellent writing, editing(photo/video/text), presentation, and communication skills, ability to work nights and weekends as needed"
    }
    """,
    """
    {
        "position_title": "Technical Project Manager",
        "location": "Santa Monica, CA",
        "work_arrangement": "On-site",
        "experience": "4-7 years of experience in project management",
        "employment_type": "Full-time",
        "pay": "$78,000 - $150,000",
        "degree": "Bachelor's degree in a hardware or software engineering field",
        "certification": "PMP or similar certification",
        "required_skills": "Ability to effectively communicate project milestones, status, and risks at all levels of the organization, delivering consumer, enterprise, or industrial products, experience managing all steps of the product lifecycle from concept to manufacturing, experience simultaneously managing different projects with multiple stakeholders, demonstrated experience developing or deploying applications of interactive, AR, VR, or XR, experience as a scrum master or similar agile processes, experience interfacing with test teams, experience working on defense & aerospace products"
    }
    """,
    """
    {
        "position_title": "Logistics Coordinator",
        "location": "Henderson, NV 89074",
        "work_arrangement": "On-site",
        "experience": "previous experience in operations or a related field is a plus",
        "employment_type": "Full-time, Shift",
        "pay": "From $14 an hour",
        "degree": "High school diploma/GED",
        "certification": "Not specified",
        "required_skills": "Excellent organizational and inter-personal and communication skills, strong organizational and multitasking abilities, proficiency in Microsoft Office and other relevant software"
    }
    """,
    """
    {
        "position_title": "Sr. Net Developer with AWS",
        "location": "Irving, TX",
        "work_arrangement": "On-site",
        "experience": "Experience with microservices, cloud services, especially with AWS, 2+ years of experience",
        "employment_type": "Full-time, Contract 12+ Months",
        "pay": "Not specified",
        "degree": "Bachelor's/Master's in computer science or related fields",
        "certification": "AWS certified(associate or professional)",
        "required_skills": "proficiency with C#, .NET Core framework 2.x & higher, Git, SVN, MongoDB, Cassandra, familiarity with dev-ops software development methods and Docker container related technologies"
    }
    """
]

In [18]:
test_examples_list = [
    '{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": "Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and price quotes to the DOD", "required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}',
    '{"position_title": "Penetration Tester", "location": "Washington, DC", "work_arrangement": "On-site", "experience": "10+ years of Penetration Testing experience", "employment_type": "Full time", "pay": "Not specified", "degree": "Bachelors Degree in Computer Science", "certification": "Offensive Security certification (OSCP, OSCE), GIAC certification (GPEN, GWAPT, GXPN), or technology specific certification (MCSE, LPIC, CCNA)", "required_skills": "NIST guidance, FedRAMP control baseline, industry best practice"}',
    '{"position_title": "NURSES - RNs or LPNs", "location": "Sudbury, MA 01776", "work_arrangement": "on-site", "experience": "Minimum of 1 year Long term care experience/SNF experience preferred", "employment_type": "full-time", "pay": "HOURLY - EVERY OTHER WEEKEND REQUIRED", "degree": "Must have a valid MA Nursing License", "certification": "RN or LPN License in Massachusetts", "required_skills": "Medication pass, treatments, resident care"}',
    '{"position_title": "Planner IV - Transportation Planner", "location": "Yakima, WA, 98901", "work_arrangement": "On-site", "experience": "5 years of increasingly responsible professional experience", "employment_type": "Full-Time", "pay": "$39.84 - $50.53 Hourly", "degree": "Bachelor\'s Degree in Planning or other related field", "certification": "None specified", "required_skills": "Transportation planning, coordination with the Yakama Nation, preparation of loans and grants"}',
    '{"position_title": "Associate Attorney", "location": "McAllen, TX", "work_arrangement": "on-site", "experience": "None specified.", "employment_type": "full-time", "pay": "$50,000 a year", "degree": "Law doctoral degree", "certification": "Admission to the state bar and in good standing with the relevant jurisdiction.", "required_skills": "Interest in Family and Criminal Law, proven track record of successful hearing coverage and strong advocacy skills."}',
    '{"position_title": "EMT-Advanced-Emergency Medical Service", "location": "Rosenberg, TX 77471", "work_arrangement": "On-site", "experience": "Pre-hospital experience preferred, experience in a high performance ALS system", "employment_type": "Full time", "pay": "$2,008.28 - $2,421.41 biweekly", "degree": "High school diploma/GED", "certification": "paramedic certification or EMS degree, AEMT, enrolled in an EMT Paramedic Program, DSHS EMT-Advanced, valid Texas driver\'s license", "required_skills": "Strong verbal and written communication, organizational skills, interpersonal skills, judgment, reasoning, decision-making, teaching"}',
    '{"position_title": "Transportation Environmental Resources Specialist", "location": "Weston, West Virginia 26452-8289", "work_arrangement": "On-site", "experience": "24 Months", "employment_type": "Full time Permanent", "pay": "$1,700.00 - $2,521.15 Biweekly", "degree": "Bachelor\'s degree from a regionally accredited college or university with a major in archeology, chemistry, geology, history, physics, geography, biology, economics, engineering, environmental studies, natural science, or a related field.", "certification": "Drivers license, DL", "required_skills": "Full-performance level, complex professional work in a specialty area in the acquisition, preservation, management and protection of the state\'s environmental/natural resources."}',
    '{"position_title": "Hotel Front Desk Clerk", "location": "La Quinta Inn & Suites, USF Tampa, FL", "work_arrangement": "On-site", "experience": "At least one year of hospitality industry experience", "employment_type": "Full Time", "pay": "$14 hourly", "degree": "High school diploma or GED", "certification": "None specified", "required_skills": "Customer service, Microsoft Office, organizational skills, communication, time management"}',
    '{"position_title": "Psychotherapist", "location": "Asbury, NJ", "work_arrangement": "On-site", "experience": "1 year", "employment_type": "Hourly", "pay": "$65 - $95 an hour", "degree": "Doctor of Psychology Doctoral degree or equivalent", "certification": "LSW Social Work License, LCSW, LPC, LAC, or other relevant licenses", "required_skills": "experience with children, strong interpersonal skills, ability to establish rapport with clients"}',
    '{"position_title": "Cryptocurrency / FX Trader - Entry Level", "location": "Not specified", "work_arrangement": "Remote", "experience": "No prior experience required", "employment_type": "Full-time or part-time", "pay": "Results-based commissions and performance bonuses", "degree": "Bachelor\'s degree in finance, economics, or related field preferred", "certification": "None specified", "required_skills": "Strong analytical skills, quick decision-making"}'
]

In [15]:
train_examples = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\Manual Labeling - Sheet1.csv")

In [16]:
train_results = train_example_list
train_contents = list(train_examples['body'])
train_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(train_contents, train_results)]
trainset=train_examples_list
trainset = [x.with_inputs('context') for x in trainset]

In [19]:
test_results = test_examples_list
test_contents = list(test_examples[1])
test_examples_list = [dspy.Example(context=content, answer=result) for content, result in zip(test_contents, test_results)]
testset = test_examples_list
testset = [x.with_inputs('context') for x in testset]

#### Test on Test set

In [20]:
answ = testset[0]
print(answ.answer)

{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": 
"Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A 
BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and price quotes to the DOD", 
"required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}

In [21]:
with dspy.context(lm=llm):
    pred = uncompiled_fs(context=testset[0].context)
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))


c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{
    'position_title': 'Federal Sales Engineer',
    'location': 'Walpole, MA',
    'work_arrange': 'Hybrid',
    'experience': 'Experience in the development and selling of power electronics or similar hardware, along with 
providing technical support, to the DOD, homeland security, prime contractors, and commercial industries (telecom, 
medical, robotics, energy, etc.)',
    'employment_type': 'Full Time',
    'pay': '$120k/year',
    'degree': 'BS in the engineering field',
    'certifications': 'Working knowledge of ERP (Epicor is preferred)',
    'required_skills': 'Inside/outside technical sales experience, experience working with outside sales reps, RFQ 
experience and price quotes to the DOD preferred, working knowledge of ERP (Epicor is preferred), familiar with 
power electronics and has sales experience in the military and industrial sectors, military service is considered a
plus'
}

In [22]:
validate_ans(answ, pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{'position_title': 'federal sales engineer', 'location': 'walpole, ma', 'work_arrange': 'hybrid', 'experience': 
'experience in the development and selling of power electronics or similar hardware, along with providing technical
support, to the dod, homeland security, prime contractors, and commercial industries (telecom, medical, robotics, 
energy, etc.)', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'bs in the engineering field', 
'certifications': 'working knowledge of erp (epicor is preferred)', 'required_skills': 'inside/outside technical 
sales experience, experience working with outside sales reps, rfq experience and price quotes to the dod preferred,
working knowledge of erp (epicor is preferred), familiar with power electronics and has sales experience in the 
military and industrial sectors, military service is considered a plus'}

0.3632122341251358

0.3632122341251358

#### Train Compiled Module
    * Need more GPU memory to Compile 

In [1]:
teleprompter = BootstrapFewShot(metric=validate_ans)
compiled_info_extract = teleprompter.compile(uncompiled_fs, trainset=trainset)

In [2]:
with dspy.context(lm=llm):
    pred = compiled_info_extract(context=testset[0].context)
    torch.cuda.empty_cache()
    gc.collect()
    print(json.loads(pred.answer.split('Answer: ')[1]))

In [ ]:
validate_ans(answ, pred)

#### Evaluate Uncompiled Quantized Unsloth

In [61]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10)

prev_score=evaluation(uncompiled_fs, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{'position_title': 'federal sales engineer', 'location': 'walpole, ma', 'work_arrange': 'hybrid', 'experience': 
'experience in the development and selling of power electronics or similar hardware, along with providing technical
support, to the dod, homeland security, prime contractors, and commercial industries (telecom, medical, robotics, 
energy, etc.)', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'bs in the engineering field', 
'certifications': 'working knowledge of erp (epicor is preferred)', 'required_skills': 'inside/outside technical 
sales experience, experience working with outside sales reps, rfq experience and price quotes to the dod preferred,
working knowledge of erp (epicor is preferred), familiar with power electronics and has sales experience in the 
military and industrial sectors, military service is considered a plus'}

0.3632122341251358

...

{"position_title": "penetration tester", "location": "washington, dc", "work_arrangement": "on-site", "experience":
"10+ years of penetration testing experience", "employment_type": "full time", "pay": "not specified", "degree": 
"bachelors degree in computer science", "certification": "offensive security certification (oscp, osce), giac 
certification (gpen, gwapt, gxpn), or technology specific certification (mcse, lpic, ccna)", "required_skills": 
"nist guidance, fedramp control baseline, industry best practice"}

{'position_title': 'penetration tester', 'location': 'washington, dc', 'work_arrangement': 'on-site', 'experience':
'10+ years', 'employment_type': 'full-time', 'pay': 'not specified', 'degree': "bachelor's degree in computer 
science", 'certifications': 'offensive security certifications (oscp, osce), giac certifications (gpe, gwapt, 
gxpn), or technology specific certifications (mcse, lpic, ccna)', 'required_skills': "knowledge of nist guidance, 
fedramp control baseline, industry best practices, and the internal revenue service (irs) publication 1075; 
experience conducting security and network audits to evaluate how well an organization's system conforms to a set 
of established criteria"}

0.6100106923282544

Average Metric: 0.9732229264533903 / 2  (48.7):  20%|██        | 2/10 [00:47<03:07, 23.49s/it]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [88]:
len(llm.history)

56